# EDA Silver Indicateur 1

In [ ]:
from pathlib import Path

import folium
import pandas as pd
from folium.plugins import MarkerCluster
from shapely import wkb
from shapely.geometry import mapping

def resolve_silver_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / 'Indicateur_1' / 'silver',
        Path.cwd().parent / 'Indicateur_1' / 'silver',
    ]
    for candidate in candidates:
        if (candidate / 'eclairage-public.parquet').exists():
            return candidate
    raise FileNotFoundError('Impossible de localiser le dossier silver de lindicateur 1')

SILVER_DIR = resolve_silver_dir()
PARIS_CENTER = [48.8566, 2.3522]
PARIS_ZOOM = 11

def create_base_map(title: str):
    m = folium.Map(location=PARIS_CENTER, zoom_start=PARIS_ZOOM, tiles='CartoDB positron')
    folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
    folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)
    return m

def add_point_layer(m, df, lat_col='latitude', lon_col='longitude', layer_name='Points', color='#1f77b4'):
    point_df = df.dropna(subset=[lat_col, lon_col]).copy()
    point_df[lat_col] = pd.to_numeric(point_df[lat_col], errors='coerce')
    point_df[lon_col] = pd.to_numeric(point_df[lon_col], errors='coerce')
    point_df = point_df.dropna(subset=[lat_col, lon_col])

    cluster = MarkerCluster(name=layer_name).add_to(m)
    for _, row in point_df.iterrows():
        tooltip_parts = []
        for column in ['nom', 'type', 'statut', 'arrondissement', 'lib_ouvrag', 'name']:
            if column in row and pd.notna(row[column]):
                tooltip_parts.append(f"{column}: {row[column]}")
        tooltip = ' | '.join(tooltip_parts) if tooltip_parts else layer_name
        folium.CircleMarker(
            location=[float(row[lat_col]), float(row[lon_col])],
            radius=3,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            tooltip=tooltip,
        ).add_to(cluster)

def add_shape_layer(m, df, geom_col='geo_shape', layer_name='Zones', color='#f28e2b'):
    if geom_col not in df.columns:
        return

    features = []
    for _, row in df.iterrows():
        value = row.get(geom_col)
        if value is None or pd.isna(value):
            continue
        try:
            geometry = wkb.loads(bytes(value))
        except Exception:
            continue

        properties = {}
        for column in ['nom', 'type', 'statut', 'arrondissement', 'name']:
            if column in row and pd.notna(row[column]):
                properties[column] = row[column]

        features.append({
            'type': 'Feature',
            'geometry': mapping(geometry),
            'properties': properties,
        })

    if not features:
        return

    folium.GeoJson(
        {'type': 'FeatureCollection', 'features': features},
        name=layer_name,
        style_function=lambda feature: {
            'color': color,
            'weight': 2,
            'fillColor': color,
            'fillOpacity': 0.25,
        },
        tooltip=folium.GeoJsonTooltip(fields=list(features[0]['properties'].keys())) if features[0]['properties'] else None,
    ).add_to(m)

def finalize_map(m):
    folium.LayerControl(collapsed=False).add_to(m)
    return m

In [ ]:
df = pd.read_parquet(SILVER_DIR / 'eclairage-public.parquet')
df = df.copy()
df['longitude'] = pd.to_numeric(df['x_wgs84'], errors='coerce')
df['latitude'] = pd.to_numeric(df['y_wgs84'], errors='coerce')
m = create_base_map('Eclairage public')
add_point_layer(m, df, lat_col='latitude', lon_col='longitude', layer_name='Points', color='#1f77b4')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / 'ilots-de-fraicheur-equipements-activites.parquet')
m = create_base_map('Ilots de fraicheur - equipements et activites')
add_point_layer(m, df, layer_name='Points', color='#2ca02c')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / 'ilots-de-fraicheur-espaces-verts-frais.parquet')
m = create_base_map('Ilots de fraicheur - espaces verts frais')
add_point_layer(m, df, layer_name='Points', color='#2ca02c')
add_shape_layer(m, df, layer_name='Zones', color='#9467bd')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / 'les-arbres.parquet')
m = create_base_map('Les arbres')
add_point_layer(m, df, layer_name='Points', color='#228b22')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / 'sanisettesparis.parquet')
m = create_base_map('Sanisettes Paris')
add_point_layer(m, df, layer_name='Points', color='#d62728')
finalize_map(m)
m

In [ ]:
df = pd.read_parquet(SILVER_DIR / 'zones-touristiques-internationales.parquet')
m = create_base_map('Zones touristiques internationales')
add_point_layer(m, df, layer_name='Points', color='#8c564b')
add_shape_layer(m, df, layer_name='Zones', color='#e377c2')
finalize_map(m)
m